# 02 · BERTimbau no Google Colab

Treina e avalia o BERTimbau nas duas tarefas do projeto e publica os modelos no Hugging Face Hub.

**Antes de começar**
1. *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4.*
2. No painel 🔑 **Secrets** (barra lateral esquerda), crie o segredo `HF_TOKEN` com um token **de escrita**
   do Hugging Face e ative o acesso deste notebook. O token nunca aparece na tela nem vai para o Git.
3. Tenha à mão o arquivo `data/processed/reviews.parquet` do seu PC.

**Regra de honestidade:** cada tarefa roda com `--teste` **uma única vez**. Se precisar repetir, conserte o
problema olhando só a validação; não ajuste nada com base no número do teste.

Tempo aproximado numa T4: 15 a 25 min para sentimento e menos de 10 min para problemas.

In [ ]:
# 1. Confere a GPU
import torch

assert torch.cuda.is_available(), "Sem GPU: ative a T4 em Ambiente de execução → Alterar o tipo."
print(torch.cuda.get_device_name(0))

In [ ]:
# @title 2. Baixa o código do projeto { display-mode: "form" }
import os
import subprocess

BRANCH = "main"  # @param {type:"string"}
REPO = "https://github.com/gabrielmarcone/analisa.ai-analisador-avaliacoes.git"
PROJECT = "/content/projeto"
if not os.path.isdir(PROJECT):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, PROJECT], check=True)
os.chdir(PROJECT)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# 3. Instala o projeto (sem mexer no PyTorch nem no pandas do Colab)
%pip install -q -e . --no-deps
%pip install -q -r requirements-bert.txt

## 4. Envia a base processada

Clique em **Escolher arquivos** e selecione `data/processed/reviews.parquet` do seu PC
(pasta do projeto). A célula confere se a base bate com a divisão versionada no Git.

In [ ]:
from pathlib import Path

import pandas as pd
from google.colab import files

target = Path("data/processed/reviews.parquet")
if not target.is_file():
    uploaded = files.upload()
    if list(uploaded) != ["reviews.parquet"]:
        raise SystemExit(f"Envie só o arquivo reviews.parquet (recebi {list(uploaded)}).")
    target.parent.mkdir(parents=True, exist_ok=True)
    Path("reviews.parquet").replace(target)

base = pd.read_parquet(target, columns=["review_id", "split", "texto", "sentimento"])
splits = pd.read_csv("splits/olist_splits_v1.csv")
merged = splits.merge(base, on="review_id", how="outer", suffixes=("_git", "_base"), indicator=True)
assert (merged["_merge"] == "both").all(), "A base e a divisão do Git têm avaliações diferentes."
assert (merged["split_git"] == merged["split_base"]).all(), "A base não usa a divisão versionada."
print(f"Base conferida: {len(base):,} avaliações".replace(",", "."))
print(base["split"].value_counts().to_string())

## 5. Sentimento (negativa × positiva)

In [ ]:
!python -m reviews.bert sentimento --teste

## 6. Tipos de problema (multirrótulo, 1.500 avaliações anotadas)

In [ ]:
!python -m reviews.bert problemas --teste

## 7. Tabela comparativa

Baseline × BERTimbau, gerada a partir de `reports/resultados.json`.

In [ ]:
from IPython.display import Markdown

Markdown(Path("reports/resultados.md").read_text(encoding="utf-8"))

## 8. Publica no Hugging Face Hub

Cria (ou atualiza) `SEU_USUARIO/bertimbau-olist-sentimento` e `SEU_USUARIO/bertimbau-olist-problemas`,
cada um com o model card gerado a partir dos resultados.

In [ ]:
# @title Publicar { display-mode: "form" }
import os
import sys

from google.colab import userdata

HF_USUARIO = ""  # @param {type:"string"}

sys.path.insert(0, "src")  # o kernel aberto ainda não enxerga o pacote recém-instalado
from reviews.bert import publish  # noqa: E402

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # lido do painel Secrets; não é impresso
publish(HF_USUARIO.strip())

## 9. Baixa os resultados para o seu PC

Baixa `resultados_bertimbau.zip`. No PC, extraia **na pasta do projeto**, substituindo os arquivos
de `reports/`. Os modelos ficam no Hugging Face; eles não vão para o Git.

In [ ]:
import zipfile

outputs = [
    "reports/resultados.json",
    "reports/resultados.md",
    "reports/bertimbau_sentimento_historico.csv",
    "reports/bertimbau_problemas_historico.csv",
]
missing = [p for p in outputs if not Path(p).is_file()]
assert not missing, f"Faltam arquivos: {missing}"
with zipfile.ZipFile("resultados_bertimbau.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for p in outputs:
        zf.write(p)
files.download("resultados_bertimbau.zip")